In [1]:
# ===========================
# Step 1: Install Dependencies and Mount Google Drive
# ===========================

# Install DeepLabCut (specific version) and TensorFlow
!pip install deeplabcut==2.3.10 tensorflow==2.12 tensorpack

# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# ===========================
# Step 2: Import libraries
# ===========================

# ===========================
# These libraries are essential for your tasks:
# TensorFlow and DeepLabCut are for running predictions.
# Matplotlib for plotting images and keypoints.
# Pandas for reading and working with .h5 files that contain prediction data.
# Numpy for calculating distances between keypoints.
# TQDM for showing progress during loops.
# ===========================

# Import necessary libraries
import tensorflow as tf
import deeplabcut
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter
import shutil

# Check TensorFlow version
print("TensorFlow version:", tf.__version__)

Loading DLC 2.3.10...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)
TensorFlow version: 2.12.0


In [3]:
# ===========================
# Step 3: Define project paths
# ===========================
project_folder_path = Path("/content/drive/My Drive/MacMillan Lab/chapter_1/measure_your_cricket/cricket_measurements_v1-emily-2025-01-13")
config_file_path = project_folder_path / 'config.yaml'

# Define folder structure
photo_folder = project_folder_path /'measure-your-cricket/put-photos-here'
results_folder = project_folder_path / 'measure-your-cricket/results'
annotated_images_folder = results_folder / 'annotated-images'
manual_review_folder = results_folder / 'flagged-for-manual-review'


# Ensure paths exist
print(f"Config file path: {config_file_path}")
print(f"Image folder path: {photo_folder}")

Config file path: /content/drive/My Drive/MacMillan Lab/chapter_1/measure_your_cricket/cricket_measurements_v1-emily-2025-01-13/config.yaml
Image folder path: /content/drive/My Drive/MacMillan Lab/chapter_1/measure_your_cricket/cricket_measurements_v1-emily-2025-01-13/measure-your-cricket/put-photos-here


In [4]:
# ===========================
# Step 4: Check your image sizes
# ===========================
from PIL import Image
from pathlib import Path
from collections import Counter

# Path to the folder with your images
folder_path = Path("/content/drive/My Drive/MacMillan Lab/chapter_1/measure_your_cricket/cricket_measurements_v1-emily-2025-01-13/measure-your-cricket/put-photos-here")

# Dictionary to store image dimensions
dimensions = []

# Loop through all images in the folder
for img_file in folder_path.iterdir():
    if img_file.suffix.lower() in [".png", ".jpg", ".jpeg"]:  # Supported image formats
        try:
            with Image.open(img_file) as img:
                dimensions.append(img.size)  # (width, height)
        except Exception as e:
            print(f"Error reading {img_file.name}: {e}")

# Count occurrences of each dimension
dimension_counts = Counter(dimensions)

# Print the results
print(f"Total images checked: {len(dimensions)}")
print("Image size counts:")
for dim, count in dimension_counts.items():
    print(f"Size {dim}: {count} images")

Total images checked: 24
Image size counts:
Size (866, 487): 24 images


In [5]:
# ===========================
# Step 5: Run DeepLabCut to Analyze Images
# ===========================
deeplabcut.analyze_time_lapse_frames(config_file_path, photo_folder, frametype='.png', shuffle=1, save_as_csv=True)
print("DeepLabCut has finished analyzing the images.")

Using snapshot-50000 for model /content/drive/My Drive/MacMillan Lab/chapter_1/measure_your_cricket/cricket_measurements_v1-emily-2025-01-13/dlc-models/iteration-1/cricket_measurements_v1Jan13-trainset95shuffle1


/usr/local/lib/python3.11/dist-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Analyzing all frames in the directory:  /content/drive/My Drive/MacMillan Lab/chapter_1/measure_your_cricket/cricket_measurements_v1-emily-2025-01-13/measure-your-cricket/put-photos-here
Starting to extract posture
Overall # of frames:  24  found with (before cropping) frame dimensions:  866 487


 83%|████████▎ | 20/24 [00:26<00:05,  1.31s/it]

Saving results in /content/drive/My Drive/MacMillan Lab/chapter_1/measure_your_cricket/cricket_measurements_v1-emily-2025-01-13/measure-your-cricket/put-photos-here...
Saving csv poses!
The folder was analyzed. Now your research can truly start!
If the tracking is not satisfactory for some frame, consider expanding the training set.
DeepLabCut has finished analyzing the images.



/usr/local/lib/python3.11/dist-packages/deeplabcut/utils/auxiliaryfunctions.py:402: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  DataMachine.to_hdf(dataname, "df_with_missing", format="table", mode="w")


In [6]:
# ===========================
# Step 6: Load predictions (from .h5 file)
# ===========================
h5_file = next(photo_folder.glob('*.h5'))
df = pd.read_hdf(h5_file)

df = df.droplevel(level=0, axis=1)  # Remove unnecessary levels
df.head(3)

bodyparts         eye_right                           eye_left              \
coords                    x           y likelihood           x           y   
10_06052025.png  578.070996  350.523697   0.999912  581.612844  216.109747   
11_06052025.png  281.405767   44.329499   0.999356  189.863533  153.405681   
12_06052025.png  172.611159  279.760592   0.978186  256.319344  398.638013   

bodyparts                  thorax_right                         horax_left  \
coords          likelihood            x           y likelihood           x   
10_06052025.png   0.999813   490.417860  342.094574   0.998581  490.151934   
11_06052025.png   0.999513   351.172459  102.768810   0.999556  260.043094   
12_06052025.png   0.971515   258.597241  227.709936   0.203617  355.794190   

bodyparts                               thorax_top                         \
coords                    y likelihood           x           y likelihood   
10_06052025.png  218.408751   0.998993  521.704216  279.525654   0.998913   
11_06052025.png  208.698113   0.989999  282.411149  135.381461   0.999162   
12_06052025.png  164.858376   0.050290  209.026354  271.327585   0.005133   

bodyparts       thorax_bottom                         
coords                      x           y likelihood  
10_06052025.png    446.362328  275.596848   0.999956  
11_06052025.png    340.610478  189.714489   0.999704  
12_06052025.png    328.937070  264.304942   0.977101

In [11]:
!pip install XlsxWriter
import xlsxwriter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 11.4 MB/s eta 0:00:00


In [14]:
# filepath: scripts/calculate_distances.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import xlsxwriter

# Setup the output folder for saving results
output_folder = results_folder / 'annotated-images'
output_folder.mkdir(parents=True, exist_ok=True)

manual_review_folder = results_folder / 'flagged-for-manual-review'
manual_review_folder.mkdir(parents=True, exist_ok=True)

measurements_df = pd.DataFrame()

measurement_pairs = {
    'eye_width': ['eye_right', 'eye_left'],
    'thorax_width': ['thorax_right', 'horax_left'],
    'thorax_length': ['thorax_top', 'thorax_bottom']
}

legend_order = ['eye_right', 'eye_left', 'thorax_right', 'horax_left', 'thorax_top', 'thorax_bottom']

keypoint_colors = {
    'eye_right': '#D93737',
    'eye_left': '#E26A1D',
    'thorax_right': '#F0C30F',
    'horax_left': '#4C8C5E',
    'thorax_top': '#2E2F8F',
    'thorax_bottom': '#9851C9'
}

def draw_black_line(x, y, ax):
    ax.plot(x, y, color='black', linewidth=1.5, alpha=0.9, zorder=1)

confidence_flags = {}
confidence_values = {}

for image_name in tqdm(df.index):
    image_path = photo_folder / image_name
    image = plt.imread(image_path)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)

    row = df.loc[image_name]
    low_confidence = False
    measurements = {}
    confidences = {}
    plotted_keypoints = {}

    for measurement, (bp1, bp2) in measurement_pairs.items():
        try:
            x0, y0, c0 = row.loc[bp1, ['x', 'y', 'likelihood']]
            x1, y1, c1 = row.loc[bp2, ['x', 'y', 'likelihood']]
            draw_black_line([x0, x1], [y0, y1], ax)
            distance = np.sqrt((x1 - x0) ** 2 + (y1 - y0) ** 2)
            measurements[measurement] = distance
            if c0 < 0.95 or c1 < 0.95:
                confidence_flags.setdefault(image_name, {})[measurement] = True
                low_confidence = True
        except KeyError:
            continue

    for bp in legend_order:
        try:
            x, y, conf = row.loc[bp, ['x', 'y', 'likelihood']]
            confidences[f'{bp}_conf'] = conf
            face_color = keypoint_colors.get(bp, 'gray')
            handle = ax.scatter(x, y, c=face_color, edgecolors='black', s=60, linewidth=1.2, zorder=2,
                                label=f"{bp.replace('_', ' ').title()} ({conf:.3f})")
            plotted_keypoints[bp] = handle
        except KeyError:
            confidences[f'{bp}_conf'] = np.nan
            continue

    ordered_handles = [plotted_keypoints[bp] for bp in legend_order if bp in plotted_keypoints]
    ordered_labels = [f"{bp.replace('_', ' ').title()} ({row.loc[bp, 'likelihood']:.3f})"
                      for bp in legend_order if bp in plotted_keypoints]
    ax.legend(ordered_handles, ordered_labels, loc="upper right", fontsize=10)

    annotated_save_path = output_folder / image_name
    plt.savefig(annotated_save_path)
    plt.close()

    if low_confidence:
        manual_review_path = manual_review_folder / image_name
        with open(annotated_save_path, 'rb') as src, open(manual_review_path, 'wb') as dst:
            dst.write(src.read())

    all_values = {**measurements, **confidences}
    measurements_df = pd.concat([measurements_df, pd.DataFrame(all_values, index=[image_name])])

# Save as Excel with red formatting for low-confidence values
csv_path = output_folder / 'measurements.csv'
excel_path = str(csv_path).replace('.csv', '.xlsx')

with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    measurements_df.to_excel(writer, sheet_name="Measurements")

    workbook = writer.book
    worksheet = writer.sheets["Measurements"]
    red_text_format = workbook.add_format({'font_color': '#FF0000'})

    for row_idx, (image_name, row) in enumerate(measurements_df.iterrows(), start=1):
        for col_idx, col_name in enumerate(measurements_df.columns, start=1):
            value = row[col_name]
            format_to_apply = None
            if col_name in measurement_pairs and confidence_flags.get(image_name, {}).get(col_name, False):
                format_to_apply = red_text_format
            if col_name.endswith('_conf') and value < 0.95:
                format_to_apply = red_text_format
            worksheet.write(row_idx, col_idx, value, format_to_apply if format_to_apply else None)

print("Measurements and images have been saved successfully.")


100%|██████████| 24/24 [00:12<00:00,  1.91it/s]

Measurements and images have been saved successfully.
